In [2]:
import pandas as pd

df = pd.read_csv("/content/Plant_1_Generation_Data.csv")
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"], dayfirst=True)
df = df.sort_values("DATE_TIME")

print(df.shape)
print(df.columns.tolist())
print(df.head())

(68778, 7)
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']
    DATE_TIME  PLANT_ID       SOURCE_KEY  DC_POWER  AC_POWER  DAILY_YIELD  \
0  2020-05-15   4135001  1BY6WEcLGh8j5v7       0.0       0.0          0.0   
20 2020-05-15   4135001  zVJPv84UY57bAof       0.0       0.0          0.0   
19 2020-05-15   4135001  zBIq5rxdHJRwDNY       0.0       0.0          0.0   
18 2020-05-15   4135001  z9Y9gH1T5YWrNuG       0.0       0.0          0.0   
17 2020-05-15   4135001  wCURE6d3bPkepu2       0.0       0.0          0.0   

    TOTAL_YIELD  
0     6259559.0  
20    7116151.0  
19    6339380.0  
18    7007866.0  
17    6782598.0  


In [3]:
# Drop nulls
df = df.dropna()

# Pick one inverter (one "user")
device_id = df["SOURCE_KEY"].iloc[0]
data = df[df["SOURCE_KEY"] == device_id].copy()

print(f"Device: {device_id}")
print(f"Readings: {len(data)}")
print(f"Period: {data['DATE_TIME'].min()} → {data['DATE_TIME'].max()}")
print(f"Total yield: {data['DAILY_YIELD'].max()} kWh")

Device: 1BY6WEcLGh8j5v7
Readings: 3154
Period: 2020-05-15 00:00:00 → 2020-06-17 23:45:00
Total yield: 8268.0 kWh


In [4]:
EMISSION_FACTOR = 0.82      # CEA India kg CO2 per kWh
CREDIT_THRESHOLD_KWH = 50   # lowered for demo (real = 1219.5)

total_kwh = 0
credits = []
credits_minted = 0

for _, row in data.iterrows():
    kwh = float(row["DAILY_YIELD"]) / 96  # 96 readings per day
    total_kwh += kwh
    co2_kg = round(total_kwh * EMISSION_FACTOR, 3)

    new_credits = int(total_kwh / CREDIT_THRESHOLD_KWH)
    if new_credits > credits_minted:
        credits_minted = new_credits
        credits.append({
            "credit_id": credits_minted,
            "device_id": device_id,
            "timestamp": str(row["DATE_TIME"]),
            "total_kwh": round(total_kwh, 3),
            "co2_avoided_kg": co2_kg,
            "period_start": str(data["DATE_TIME"].iloc[0]),
            "period_end": str(row["DATE_TIME"]),
            "methodology": "CEA Grid Emission Factor 0.82 kg/kWh",
            "standard": "CTN-SOLAR-V1",
            "location": "India"
        })
        print(f"Credit #{credits_minted} → {round(total_kwh, 2)} kWh → {co2_kg} kg CO2 avoided")

print(f"\nTotal: {round(total_kwh, 2)} kWh | {credits_minted} credits | {round(total_kwh * EMISSION_FACTOR, 2)} kg CO2")

Credit #1 → 58.65 kWh → 48.095 kg CO2 avoided
Credit #2 → 115.5 kWh → 94.714 kg CO2 avoided
Credit #3 → 152.35 kWh → 124.923 kg CO2 avoided
Credit #4 → 220.39 kWh → 180.72 kg CO2 avoided
Credit #5 → 277.42 kWh → 227.482 kg CO2 avoided
Credit #6 → 309.31 kWh → 253.637 kg CO2 avoided
Credit #7 → 378.15 kWh → 310.081 kg CO2 avoided
Credit #8 → 415.05 kWh → 340.344 kg CO2 avoided
Credit #9 → 453.88 kWh → 372.178 kg CO2 avoided
Credit #10 → 538.54 kWh → 441.602 kg CO2 avoided
Credit #11 → 584.35 kWh → 479.17 kg CO2 avoided
Credit #12 → 631.53 kWh → 517.851 kg CO2 avoided
Credit #13 → 680.46 kWh → 557.977 kg CO2 avoided
Credit #14 → 730.88 kWh → 599.321 kg CO2 avoided
Credit #15 → 783.13 kWh → 642.167 kg CO2 avoided
Credit #16 → 836.65 kWh → 686.052 kg CO2 avoided
Credit #17 → 891.6 kWh → 731.112 kg CO2 avoided
Credit #18 → 947.53 kWh → 776.973 kg CO2 avoided
Credit #20 → 1004.59 kWh → 823.762 kg CO2 avoided
Credit #21 → 1062.63 kWh → 871.354 kg CO2 avoided
Credit #22 → 1121.35 kWh → 919.504

In [5]:
import hashlib
import json

DEVICE_SECRET = "ctn_secret_001"

def sign_record(record):
    record_str = json.dumps(record, sort_keys=True)
    signature = hashlib.sha256(
        (record_str + DEVICE_SECRET).encode()
    ).hexdigest()
    return {**record, "signature": signature}

signed_credits = [sign_record(c) for c in credits]

# Show one signed credit
print(json.dumps(signed_credits[0], indent=2))

{
  "credit_id": 1,
  "device_id": "1BY6WEcLGh8j5v7",
  "timestamp": "2020-05-15 09:45:00",
  "total_kwh": 58.652,
  "co2_avoided_kg": 48.095,
  "period_start": "2020-05-15 00:00:00",
  "period_end": "2020-05-15 09:45:00",
  "methodology": "CEA Grid Emission Factor 0.82 kg/kWh",
  "standard": "CTN-SOLAR-V1",
  "location": "India",
  "signature": "32ebb38a7218a25a4b5d50c1a15ef557ad8a6cb98b27eb998be45faec5032b64"
}


In [6]:
from google.colab import files

# Save as JSON file
with open("ctn_credits.json", "w") as f:
    json.dump(signed_credits, f, indent=2)

# Download it
files.download("ctn_credits.json")

print(f"{len(signed_credits)} signed credit records exported")
print("Next step: upload each record to IPFS via Pinata")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1684 signed credit records exported
Next step: upload each record to IPFS via Pinata


In [7]:
import hashlib, json

DEVICE_SECRET = "ctn_secret_001"
EMISSION_FACTOR = 0.82
CREDIT_THRESHOLD_KWH = 50

total_kwh = 0
credits = []
credits_minted = 0

for _, row in data.iterrows():
    kwh = float(row["DAILY_YIELD"]) / 96
    total_kwh += kwh
    co2_kg = round(total_kwh * EMISSION_FACTOR, 3)

    new_credits = int(total_kwh / CREDIT_THRESHOLD_KWH)
    if new_credits > credits_minted:
        credits_minted = new_credits
        record = {
            "credit_id": credits_minted,
            "device_id": device_id,
            "timestamp": str(row["DATE_TIME"]),
            "total_kwh": round(total_kwh, 3),
            "co2_avoided_kg": co2_kg,
            "period_start": str(data["DATE_TIME"].iloc[0]),
            "period_end": str(row["DATE_TIME"]),
            "methodology": "CEA Grid Emission Factor 0.82 kg/kWh",
            "standard": "CTN-SOLAR-V1",
            "location": "India"
        }
        record_str = json.dumps(record, sort_keys=True)
        record["signature"] = hashlib.sha256(
            (record_str + DEVICE_SECRET).encode()
        ).hexdigest()
        credits.append(record)

with open("ctn_credits.json", "w") as f:
    json.dump(credits, f, indent=2)

from google.colab import files
files.download("ctn_credits.json")
print(f"{len(credits)} signed credits exported")
print(f"Sample:\n{json.dumps(credits[0], indent=2)}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

1684 signed credits exported
Sample:
{
  "credit_id": 1,
  "device_id": "1BY6WEcLGh8j5v7",
  "timestamp": "2020-05-15 09:45:00",
  "total_kwh": 58.652,
  "co2_avoided_kg": 48.095,
  "period_start": "2020-05-15 00:00:00",
  "period_end": "2020-05-15 09:45:00",
  "methodology": "CEA Grid Emission Factor 0.82 kg/kWh",
  "standard": "CTN-SOLAR-V1",
  "location": "India",
  "signature": "32ebb38a7218a25a4b5d50c1a15ef557ad8a6cb98b27eb998be45faec5032b64"
}


In [8]:
import requests
import json

PINATA_JWT = "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ1c2VySW5mb3JtYXRpb24iOnsiaWQiOiI4NzE5OTJhMC0xNjM4LTQ5MDQtOTk0OS04NDZjNDFkOWU2MjAiLCJlbWFpbCI6ImFxdWkuYWRpLmpAZ21haWwuY29tIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsInBpbl9wb2xpY3kiOnsicmVnaW9ucyI6W3siZGVzaXJlZFJlcGxpY2F0aW9uQ291bnQiOjEsImlkIjoiRlJBMSJ9LHsiZGVzaXJlZFJlcGxpY2F0aW9uQ291bnQiOjEsImlkIjoiTllDMSJ9XSwidmVyc2lvbiI6MX0sIm1mYV9lbmFibGVkIjpmYWxzZSwic3RhdHVzIjoiQUNUSVZFIn0sImF1dGhlbnRpY2F0aW9uVHlwZSI6InNjb3BlZEtleSIsInNjb3BlZEtleUtleSI6ImI1OTlhYmNlYzNlMTZmOTFlNWMyIiwic2NvcGVkS2V5U2VjcmV0IjoiYTA3YTljZjdiNTQ4MTI5YjA4ZmM3Y2I4ZjhjYzQ1NmZiNDU2YzQzN2Q2OWRmMjBkMmYwMmQ1YjU5ZjQyNTA2OSIsImV4cCI6MTgwNzUxMjAyM30.UO3-uvm0Uxc5P6TG0yevuqdwzIF4tFK63NMrZzMnGOk"

def upload_to_ipfs(record):
    response = requests.post(
        "https://api.pinata.cloud/pinning/pinJSONToIPFS",
        json={
            "pinataContent": record,
            "pinataMetadata": {"name": f"CTN-Credit-{record['credit_id']}"}
        },
        headers={
            "Authorization": f"Bearer {PINATA_JWT}",
            "Content-Type": "application/json"
        }
    )
    return response.json()["IpfsHash"]

# Upload first 5 credits as demo
ipfs_records = []
for credit in credits[:5]:
    ipfs_hash = upload_to_ipfs(credit)
    ipfs_records.append({
        "credit_id": credit["credit_id"],
        "ipfs_hash": ipfs_hash,
        "url": f"https://gateway.pinata.cloud/ipfs/{ipfs_hash}"
    })
    print(f"Credit #{credit['credit_id']} → {ipfs_hash}")

print("\nDone. Open any URL above in browser to verify.")

Credit #1 → QmUDq92qrsxeV7BD9tKzbfGxBYpToCvjV4N4xbFdEATeBk
Credit #2 → QmXzQtJTwtxB8uE9qFCgBkvtpfVRNPreXmPMJcxPh3Cbcy
Credit #3 → QmbpXzAavSQzZbyhYLwgRDaQ5cGzDtoD7j6shDkdWVGo64
Credit #4 → QmX5DxGWuj3DkCPLjZoLoiFHyv88HwBoqHhL8iuFZvRqT4
Credit #5 → QmYJG7keB5i2e4gBLM9c9QPAjnGfWZUXDmQugxLFtTceXF

Done. Open any URL above in browser to verify.


In [9]:
# Export first 5 credits as JS array for dashboard
import json
js_output = "const CREDITS = " + json.dumps(signed_credits[:5], indent=2) + ";"
with open("credits_data.js", "w") as f:
    f.write(js_output)
files.download("credits_data.js")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import time, json, requests
from google.colab import files

PINATA_JWT = "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJ1c2VySW5mb3JtYXRpb24iOnsiaWQiOiI4NzE5OTJhMC0xNjM4LTQ5MDQtOTk0OS04NDZjNDFkOWU2MjAiLCJlbWFpbCI6ImFxdWkuYWRpLmpAZ21haWwuY29tIiwiZW1haWxfdmVyaWZpZWQiOnRydWUsInBpbl9wb2xpY3kiOnsicmVnaW9ucyI6W3siZGVzaXJlZFJlcGxpY2F0aW9uQ291bnQiOjEsImlkIjoiRlJBMSJ9LHsiZGVzaXJlZFJlcGxpY2F0aW9uQ291bnQiOjEsImlkIjoiTllDMSJ9XSwidmVyc2lvbiI6MX0sIm1mYV9lbmFibGVkIjpmYWxzZSwic3RhdHVzIjoiQUNUSVZFIn0sImF1dGhlbnRpY2F0aW9uVHlwZSI6InNjb3BlZEtleSIsInNjb3BlZEtleUtleSI6ImI1OTlhYmNlYzNlMTZmOTFlNWMyIiwic2NvcGVkS2V5U2VjcmV0IjoiYTA3YTljZjdiNTQ4MTI5YjA4ZmM3Y2I4ZjhjYzQ1NmZiNDU2YzQzN2Q2OWRmMjBkMmYwMmQ1YjU5ZjQyNTA2OSIsImV4cCI6MTgwNzUxMjAyM30.UO3-uvm0Uxc5P6TG0yevuqdwzIF4tFK63NMrZzMnGOk"

def upload_to_ipfs(record, retries=3):
    for attempt in range(retries):
        try:
            r = requests.post(
                "https://api.pinata.cloud/pinning/pinJSONToIPFS",
                json={
                    "pinataContent": record,
                    "pinataMetadata": {"name": f"CTN-Credit-{record['credit_id']}"}
                },
                headers={"Authorization": f"Bearer {PINATA_JWT}"}
            )
            data = r.json()
            if "IpfsHash" in data:
                return data["IpfsHash"]
            print(f"Retry {attempt+1}: {data}")
            time.sleep(2)
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(2)
    return None

ipfs_records = []
failed = []

for credit in credits:
    h = upload_to_ipfs(credit)
    if h:
        ipfs_records.append({**credit, "ipfs_hash": h})
        print(f"✓ #{credit['credit_id']} → {h}")
    else:
        failed.append(credit['credit_id'])
        print(f"✗ #{credit['credit_id']} failed")
    time.sleep(0.5)  # increased delay

print(f"\nUploaded: {len(ipfs_records)} | Failed: {len(failed)}")
if failed:
    print(f"Failed IDs: {failed}")

with open("credits_with_ipfs.json", "w") as f:
    json.dump(ipfs_records, f, indent=2)

files.download("credits_with_ipfs.json")

✓ #1 → QmUDq92qrsxeV7BD9tKzbfGxBYpToCvjV4N4xbFdEATeBk
✓ #2 → QmXzQtJTwtxB8uE9qFCgBkvtpfVRNPreXmPMJcxPh3Cbcy
✓ #3 → QmbpXzAavSQzZbyhYLwgRDaQ5cGzDtoD7j6shDkdWVGo64
✓ #4 → QmX5DxGWuj3DkCPLjZoLoiFHyv88HwBoqHhL8iuFZvRqT4
✓ #5 → QmYJG7keB5i2e4gBLM9c9QPAjnGfWZUXDmQugxLFtTceXF
✓ #6 → QmYa6vC6YrnqMzBQDetra1KpATJMxU31cdptNCudfjX8pr
✓ #7 → QmXaAufH9nbyhcS6v7DWKdk7Gk7bADuQpuJV2ufRVaZwNV
✓ #8 → QmUQQLoPL862dbmFgkPUr6kQ3t5SdjwWi51f4pfs5uFnWA
✓ #9 → QmUnqV7fPecL55tDuR6mBiu1CWtGZAueP6Fjey7dVCsiV4
✓ #10 → QmVNtrhXf53pXvn8A9NDvwp45goHHAU4gY4deqKAVehKzx
✓ #11 → QmeMaTPuyLBPrk6sUXmm4Mu5F5kEizGQXLkbFgnhSK1n7i
✓ #12 → QmZqwWzdrHnuN4dgfaaVtfeJ2Ec4K1VYyTPfsqmSS5BCTL
✓ #13 → QmPHFQotDWuMv3EmoULuqLAkkwdm2f4rssg6DgU9VaVU5z
✓ #14 → QmVU6c3uAaMTtPdNx9BfweRFegauijsw4x5aXKw9ArdEGV
✓ #15 → QmZoAk7xBarxGmHnbVNwsKw1EYJdofGi9xTY9nhCHPNzm9
✓ #16 → QmTpG7fgxvbnrk6RA71ELZnD9wkMW6FP8gEEu7yacsWiyF
✓ #17 → QmYYppqnkPtVBkFAGeGTEGDdCGqq6LDgTQunrkExfq9zdZ
✓ #18 → QmcGDh4BAyFn87dCUcZY8oSw9tm7qQdZs7hmK4ug3NMx14
✓ #20 → QmdfjVmkbEQ

KeyboardInterrupt: 

In [12]:
# Upload ALL credits as one file — costs 1 API call, not 2122
bulk_record = {
    "dataset": "CTN-SOLAR-V1",
    "device_id": "1BY6WEcLGh8j5v7",
    "location": "Patna, Bihar, India",
    "total_credits": len(credits),
    "methodology": "CEA 0.82 kg/kWh",
    "credits": credits  # all 2122
}

r = requests.post(
    "https://api.pinata.cloud/pinning/pinJSONToIPFS",
    json={
        "pinataContent": bulk_record,
        "pinataMetadata": {"name": "CTN-ALL-CREDITS-PLANT1"}
    },
    headers={"Authorization": f"Bearer {PINATA_JWT}"}
)
print(r.json())

{'error': {'reason': 'FORBIDDEN', 'details': 'Account blocked due to plan usage limit'}}
